In [ ]:
import tensorflow as tf
import random
import math

def Data_Normalization_And_Augmentation(image_path, normalization_method):
    
    """Normalizes & Augments the Pixels in your Image, creating Several Prototypes"""

    base_img = tf.io.read_file(image_path)
    base_img = tf.io.decode_image(base_img, channels=3, expand_animations=False)
    base_img = tf.cast(base_img, tf.float32)

    orig_h = tf.shape(base_img)[0]
    orig_w = tf.shape(base_img)[1]

    original_image_tensor = tf.transpose(base_img, [2, 0, 1])

    scaled_img = base_img / 255.0

    means = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
    stds = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)

    if normalization_method == "Min Max Scaling":
        normalized_img = scaled_img

    elif normalization_method == "Standard Deviation":
        normalized_img = (scaled_img - means) / stds

    normalized_image_tensor = tf.transpose(normalized_img, [2, 0, 1])

    crop_scale = random.uniform(0.8, 1.0)

    crop_h = int(orig_h * crop_scale)
    crop_w = int(orig_w * crop_scale)

    start_y = random.randint(0, int(orig_h) - crop_h)
    start_x = random.randint(0, int(orig_w) - crop_w)

    cropped_img = tf.image.crop_to_bounding_box(
        base_img,
        start_y,
        start_x,
        crop_h,
        crop_w
    )

    resized_crop_img = tf.image.resize(
        cropped_img,
        [244, 244],
        method="nearest"
    )

    resized_crop_tensor = tf.transpose(resized_crop_img, [2, 0, 1])

    max_shift_x = int(orig_w * 0.1)
    max_shift_y = int(orig_h * 0.1)

    shift_x = random.randint(-max_shift_x, max_shift_x)
    shift_y = random.randint(-max_shift_y, max_shift_y)

    affine_img = tf.raw_ops.ImageProjectiveTransformV3(
        images=tf.expand_dims(base_img, 0),
        transforms=[[
            1.0, 0.0, -shift_x,
            0.0, 1.0, -shift_y,
            0.0, 0.0
        ]],
        output_shape=tf.shape(base_img)[:2],
        interpolation="NEAREST",
        fill_mode="CONSTANT",
        fill_value=0.0
    )[0]

    random_affine_tensor = tf.transpose(affine_img, [2, 0, 1])

    angle_deg = random.uniform(-15, 15)
    angle_rad = math.radians(angle_deg)

    cos_a = math.cos(angle_rad)
    sin_a = math.sin(angle_rad)

    cx = float(orig_w) / 2.0
    cy = float(orig_h) / 2.0

    rotation_img = tf.raw_ops.ImageProjectiveTransformV3(
        images=tf.expand_dims(base_img, 0),
        transforms=[[
            cos_a,
            sin_a,
            cx - cos_a * cx - sin_a * cy,
            -sin_a,
            cos_a,
            cy + sin_a * cx - cos_a * cy,
            0.0,
            0.0
        ]],
        output_shape=tf.shape(base_img)[:2],
        interpolation="NEAREST",
        fill_mode="CONSTANT",
        fill_value=0.0
    )[0]

    random_rotation_tensor = tf.transpose(rotation_img, [2, 0, 1])

    if random.random() > 0.5:
        flipped_img = tf.image.flip_left_right(base_img)
        random_horizontal_flipped_tensor = tf.transpose(flipped_img, [2, 0, 1])
    else:
        random_horizontal_flipped_tensor = original_image_tensor

    return (
        original_image_tensor,
        normalized_image_tensor,
        resized_crop_tensor,
        random_affine_tensor,
        random_rotation_tensor,
        random_horizontal_flipped_tensor
    )

In [ ]:
import tensorflow as tf
import random

def Batch_Norm_And_Dropout(batch_activations, gamma, beta, network_instance, dropout_probability=0.20, training=True):

    """Normalizes the Layer Output as Batch Normalization & Kills off the Functionality of Certain Neurons as Dropout Layers"""

    batch_size = tf.shape(batch_activations)[0]
    feature_size = tf.shape(batch_activations)[1]

    eps = 1e-5

    mean = tf.reduce_mean(batch_activations, axis=0)

    variance = tf.reduce_mean(
        tf.square(batch_activations - mean),
        axis=0
    )

    standardized_values = (batch_activations - mean) / tf.sqrt(variance + eps)

    bn_outputs = (gamma * standardized_values) + beta

    if training:
        scale_factor = 1.0 / (1.0 - dropout_probability)

        for attribute_name in vars(network_instance):

            attribute_value = getattr(network_instance, attribute_name)

            if isinstance(attribute_value, tf.Variable) or tf.is_tensor(attribute_value):

                tensor_matrix = tf.convert_to_tensor(attribute_value)

                if len(tensor_matrix.shape) == 2:

                    dropout_mask = tf.cast(
                        tf.random.uniform(tf.shape(tensor_matrix)) >= dropout_probability,
                        dtype=tensor_matrix.dtype
                    )

                    pruned_matrix = tensor_matrix * dropout_mask * scale_factor

                    setattr(network_instance, attribute_name, pruned_matrix)

    return bn_outputs


class PropertyRegressionModel:
    def __init__(self):
        self.layer1_neurons = tf.Variable(
            [[10.0, 5.0, 15.0, 20.0]],
            trainable=True
        )


my_model = PropertyRegressionModel()

raw_layer_outputs = tf.constant([
    [2.5, 4.0, 1.2, 8.5],
    [1.5, 6.0, 3.8, 11.5]
], dtype=tf.float32)

learned_gamma = tf.Variable([1.0, 1.0, 1.0, 1.0])

learned_beta = tf.Variable([0.0, 0.0, 0.0, 0.0])


print("--- Before Pipeline Run ---")
print("Model Layer 1 Neurons :\n", my_model.layer1_neurons.numpy())


normalized_output_matrix = Batch_Norm_And_Dropout(
    batch_activations=raw_layer_outputs,
    gamma=learned_gamma,
    beta=learned_beta,
    network_instance=my_model,
    dropout_probability=0.25,
    training=True
)


print("\n--- After Pipeline Run ---")
print("Returned Normalized Matrix :\n", normalized_output_matrix.numpy())

print("\nModel Layer 1 Neurons :\n", my_model.layer1_neurons.numpy())

In [ ]:
import tensorflow as tf

def Inject_Residual_Connections(network_instance, original_inputs, safety_seed):
    
    """Injects Residual Connections in your Network through Inputs"""

    for attribute_name in vars(network_instance):

        attribute_value = getattr(network_instance, attribute_name)

        if isinstance(attribute_value, tf.Variable) or tf.is_tensor(attribute_value):

            layer_neurons = tf.convert_to_tensor(attribute_value)

            batch_size = tf.shape(layer_neurons)[0]
            layer_feature_size = tf.shape(layer_neurons)[1]

            input_feature_size = tf.shape(original_inputs)[1]

            if layer_neurons.shape.rank != 2:
                continue

            if tf.equal(layer_feature_size, input_feature_size):

                updated_layer_state = layer_neurons + original_inputs

            else:

                projection_weights = tf.random.stateless_normal(
                    shape=[input_feature_size, layer_feature_size],
                    seed=[safety_seed, 0]
                )

                projected_inputs = tf.matmul(
                    original_inputs,
                    projection_weights
                )

                updated_layer_state = layer_neurons + projected_inputs

            setattr(network_instance, attribute_name, updated_layer_state)


class DeepVisionNetwork:
    def __init__(self):

        self.layer1_neurons = tf.Variable(
            [[10.0, 5.0, 15.0, 20.0, 8.0]],
            trainable=True
        )

        self.layer2_neurons = tf.Variable(
            [[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]],
            trainable=True
        )


my_vision_model = DeepVisionNetwork()

raw_image_inputs = tf.constant(
    [[2.5, 3.0, 2.0, 22.0, 45.0]],
    dtype=tf.float32
)


print("--- Before Residual Connection Injection ---")

print(
    "Layer 1 Neurons ( 5 nodes ) :\n",
    my_vision_model.layer1_neurons.numpy()
)

print(
    "Layer 2 Neurons ( 10 nodes ) :\n",
    my_vision_model.layer2_neurons.numpy()
)


Inject_Residual_Connections(
    network_instance=my_vision_model,
    original_inputs=raw_image_inputs,
    safety_seed=2
)


print("\n--- After Residual Connection Injections ---")

print(
    "Layer 1 Neurons ( Direct Addition Applied ) :\n",
    my_vision_model.layer1_neurons.numpy()
)

print(
    "Layer 2 Neurons ( Projection Arithmetic Applied ) :\n",
    my_vision_model.layer2_neurons.numpy()
)

In [ ]:
import tensorflow as tf

def Optimize_And_Schedule_Network(network_instance, initial_lr, current_epoch, step_size=10, gamma=0.1, weight_decay=1e-4, clip_value=1.0):

    """Integrates a System of Learning Rate Scheduling, Gradient Clipping & Weight Decay"""

    decay_exponent = current_epoch // step_size
    current_lr = initial_lr * (gamma ** decay_exponent)

    for attribute_name in vars(network_instance):

        attribute_value = getattr(network_instance, attribute_name)

        if isinstance(attribute_value, tf.Variable):

            param_tensor = attribute_value
            grad_tensor = param_tensor.gradient if hasattr(param_tensor, "gradient") else None

            if grad_tensor is None:
                continue

            clipped_grad = tf.clip_by_value(
                grad_tensor,
                -clip_value,
                clip_value
            )

            decayed_param = param_tensor - (
                current_lr * weight_decay * param_tensor
            )

            updated_param_tensor = decayed_param - (
                current_lr * clipped_grad
            )

            param_tensor.assign(updated_param_tensor)

    return current_lr


class DeepRegressionModel:
    def __init__(self):

        self.weight_matrix = tf.Variable(
            [[1.5, -2.0], [0.5, 3.1]],
            trainable=True
        )


my_model_instance = DeepRegressionModel()

active_epoch = 12

print("--- Before Global Optimization Pass ---")

print(
    "Raw Parameter Weights :\n",
    my_model_instance.weight_matrix.numpy()
)

pending_gradients = tf.constant(
    [[50.0, -0.01], [12.4, -45.0]],
    dtype=tf.float32
)

effective_lr = Optimize_And_Schedule_Network(
    network_instance=my_model_instance,
    initial_lr=0.1,
    current_epoch=active_epoch,
    step_size=10,
    gamma=0.5,
    weight_decay=1e-4,
    clip_value=1.0
)

print("\n--- After Global Optimization Pass ---")

print(
    f"Operational Learning Rate applied ( Decayed ) : {effective_lr}"
)

print(
    "Final Optimized Weights ( Clipped, Decayed, and Baseline Shifted ) :\n",
    my_model_instance.weight_matrix.numpy()
)

In [ ]:
import tensorflow as tf
import keras

class CustomConvolutionalArchitecture(keras.Model):

    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = keras.layers.Conv2D(
            filters=16,
            kernel_size=3,
            padding="same"
        )

        self.relu1 = keras.layers.ReLU()
        self.pool1 = keras.layers.MaxPooling2D(
            pool_size=2,
            strides=2
        )

        self.conv2 = keras.layers.Conv2D(
            filters=32,
            kernel_size=3,
            padding="same"
        )

        self.relu2 = keras.layers.ReLU()
        self.pool2 = keras.layers.MaxPooling2D(
            pool_size=2,
            strides=2
        )

        self.flatten_features = keras.layers.Flatten()

        self.fc1 = keras.layers.Dense(128)
        self.relu3 = keras.layers.ReLU()

        self.output_layer = keras.layers.Dense(num_classes)

        self.conv1_neurons = None
        self.conv2_neurons = None
        self.fc1_neurons = None
        self.output_neurons = None

    def call(self, x, training=False):

        out = self.conv1(x)
        out = self.relu1(out)
        out = self.pool1(out)

        self.conv1_neurons = out

        out = self.conv2(out)
        out = self.relu2(out)
        out = self.pool2(out)

        self.conv2_neurons = out

        out = self.flatten_features(out)

        out = self.fc1(out)
        out = self.relu3(out)

        self.fc1_neurons = out

        logits = self.output_layer(out)

        self.output_neurons = logits

        return logits

if __name__ == "__main__":

    (
        original_tensor,
        normalized_tensor,
        resized_crop_tensor,
        affine_tensor,
        rotation_tensor,
        flipped_tensor
    ) = Data_Normalization_And_Augmentation(
        "input_image.png",
        normalization_method="Standard Deviation"
    )

    print("Successfully Created 6 Image Prototypes")

    cnn_model = CustomConvolutionalArchitecture(num_classes=10)

    print("Successfully Created a Convolutional Architectural Network")

    input_batch = tf.expand_dims(
        tf.transpose(normalized_tensor, [1, 2, 0]),
        axis=0
    )

    model_predictions = cnn_model(input_batch)

    mock_activations = tf.random.normal([1, 128])

    bn_gamma = tf.ones([128])
    bn_beta = tf.zeros([128])

    normalized_batch_features = Batch_Norm_And_Dropout(
        batch_activations=mock_activations,
        gamma=bn_gamma,
        beta=bn_beta,
        network_instance=cnn_model,
        dropout_probability=0.30,
        training=True
    )

    print("Successfully Integrated Batch Normalization & Dropout Layers in your Network")

    highway_inputs = tf.random.normal([1, 128])

    Inject_Residual_Connections(
        network_instance=cnn_model,
        original_inputs=highway_inputs,
        safety_seed=2
    )

    print("Successfully Injected Residual Connections into your Network")

    with tf.GradientTape() as tape:

        predictions = cnn_model(input_batch)

        mock_loss = tf.reduce_mean(tf.square(predictions))

    gradients = tape.gradient(
        mock_loss,
        cnn_model.trainable_variables
    )

    gradient_map = {
        id(variable): gradient
        for variable, gradient in zip(
            cnn_model.trainable_variables,
            gradients
        )
    }

    active_lr = Optimize_And_Schedule_Network(
        network_instance=cnn_model,
        gradients=gradient_map,
        initial_lr=0.1,
        current_epoch=25,
        step_size=10,
        gamma=0.5,
        weight_decay=1e-4,
        clip_value=1.0
    )

    print(
        f"Successfully Integrated Learning Rate Scheduling, "
        f"Weight Decay & Gradient Clipping into your Network, "
        f"Learning Rate = {active_lr}"
    )